In [ ]:
import torch
import torch.nn as nn
from transformers import BertModel, BertTokenizer
from torchvision.models import resnet50
from torchvision import transforms
from PIL import Image

# 1. Text Encoder: Use a Pretrained Transformer (BERT)
class TextEncoder(nn.Module):
    def __init__(self, pretrained_model="bert-base-uncased"):
        super(TextEncoder, self).__init__()
        self.bert = BertModel.from_pretrained(pretrained_model)
    
    def forward(self, input_ids, attention_mask):
        # Get CLS token embedding
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask) # hidden_dim of bert = 768
        return outputs.pooler_output  # [batch_size, hidden_dim]

# 2. Image Encoder: Use a Pretrained CNN (ResNet)
class ImageEncoder(nn.Module):
    def __init__(self):
        super(ImageEncoder, self).__init__()
        self.resnet = resnet50(pretrained=True)
        self.resnet.fc = nn.Identity()  # Remove the classification head
    
    def forward(self, images):
        return self.resnet(images)  # [batch_size, feature_dim] (feature_dim=2048)

# 3. Multimodal Model: Combine Image and Text Encoders
class MultimodalModel(nn.Module):
    def __init__(self, text_hidden_dim=768, image_hidden_dim=2048, common_dim=512):
        super(MultimodalModel, self).__init__()
        self.text_encoder = TextEncoder()
        self.image_encoder = ImageEncoder()
        
        # Project both modalities to a common dimension
        self.text_projector = nn.Linear(text_hidden_dim, common_dim)
        self.image_projector = nn.Linear(image_hidden_dim, common_dim)
    
    def forward(self, input_ids, attention_mask, images):
        text_features = self.text_encoder(input_ids, attention_mask) # [batch_size, 768]
        text_features = self.text_projector(text_features)  # [batch_size, 512]
        
        image_features = self.image_encoder(images) # [batch_size, 2048]
        image_features = self.image_projector(image_features) # [batch_size, 512]
        return text_features, image_features

# 4. Training Step: Contrastive Loss
class ContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.1):
        super(ContrastiveLoss, self).__init__()
        self.temperature = temperature
        self.softmax = nn.Softmax(dim=1)
    
    def forward(self, text_features, image_features):
        # Normalize embeddings
        text_features = nn.functional.normalize(text_features, p=2, dim=1)
        image_features = nn.functional.normalize(image_features, p=2, dim=1)
        
        # Compute similarity scores
        logits = torch.matmul(text_features, image_features.T) / self.temperature
        labels = torch.arange(len(text_features)).to(text_features.device)
        
        # Cross entropy loss
        loss = nn.CrossEntropyLoss()(logits, labels)
        return loss

# 5. Data Preparation
# Example image-text pair dataset (replace with your dataset)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def preprocess_text(text):
    encoded = tokenizer(text, padding="max_length", truncation=True, max_length=32, return_tensors="pt")
    return encoded["input_ids"].squeeze(0), encoded["attention_mask"].squeeze(0)

def preprocess_image(image_path):
    image = Image.open(image_path).convert("RGB")
    return transform(image).unsqueeze(0)

# 6. Training Loop
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = MultimodalModel().to(device)
criterion = ContrastiveLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Dummy dataset
image_paths = ["image1.jpg", "image2.jpg"]  # Replace with your image paths
texts = ["a cat sitting on a mat", "a dog playing in the park"]  # Replace with your text descriptions

for epoch in range(10):  # Number of epochs
    for image_path, text in zip(image_paths, texts):
        # Preprocess data
        input_ids, attention_mask = preprocess_text(text)
        image_tensor = preprocess_image(image_path)
        
        input_ids, attention_mask, image_tensor = (
            input_ids.to(device),
            attention_mask.to(device),
            image_tensor.to(device),
        )
        
        # Forward pass
        text_features, image_features = model(input_ids.unsqueeze(0), attention_mask.unsqueeze(0), image_tensor)
        
        # Compute loss
        loss = criterion(text_features, image_features)
        
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f"Epoch {epoch + 1}, Loss: {loss.item():.4f}")

print("Training complete.")
